# PhenoCrop-Presto Ablation Study

Evaluates the individual contribution of **Sentinel-1** (SAR) and **Sentinel-2** (optical) modalities
on the PhenoCropPresto architecture across three configurations:

| Mode | Description |
|---|---|
| `s1s2` | Both modalities (baseline — matches `model_3_presto.ipynb`) |
| `s1` | S1 (SAR) only — S2 tokens are zero-masked |
| `s2` | S2 (optical) only — S1 tokens are zero-masked |

> **Usage:** Change `ABLATION_MODE` below, then **Run All**. Repeat for each mode, then run the final summary cell.

## Cell 0 — Colab Drive Mount

In [ ]:
import os
USE_COLAB_MOUNT = True
if USE_COLAB_MOUNT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Colab mount skipped: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 1 — ⚙️ Ablation Config

> **Change `ABLATION_MODE` here, then Run All.**
> - `"s1s2"` → Sentinel-1 + Sentinel-2 (full model, should match model_3_presto baseline)
> - `"s1"` → Sentinel-1 only (S2 tokens zeroed out)
> - `"s2"` → Sentinel-2 only (S1 tokens zeroed out)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║   ▼▼  CHANGE THIS TO SWITCH ABLATION MODE  ▼▼       ║
ABLATION_MODE = "s2"    # "s1s2" | "s1" | "s2"
# ╚══════════════════════════════════════════════════════╝

from pathlib import Path
DATASET_ROOT   = "/content/drive/MyDrive/pheno_crop_v2"
LOCAL_FALLBACK = Path("./dataset/pheno_crop_v2")
OUTPUT_DIR     = Path("./models")
OUTPUT_DIR.mkdir(exist_ok=True)

root = Path(DATASET_ROOT)
if not root.exists():
    if LOCAL_FALLBACK.exists():
        root = LOCAL_FALLBACK.resolve()
        print(f"Using local fallback: {root}")
    else:
        raise FileNotFoundError(f"Dataset not found at '{DATASET_ROOT}' or '{LOCAL_FALLBACK}'.")

s1_dir  = root / "sentinel_1"
s2_dir  = root / "sentinel_2"
gt_path = root / "ground_truth.csv"
print(f"gt exists: {gt_path.exists()} | Mode: {ABLATION_MODE.upper()}")

gt exists: True | Mode: S2


## Cell 2 — Imports, Seeds & Config

In [ ]:
import math, random, warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    confusion_matrix, classification_report,
)
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Feature columns (identical to model_3_presto.ipynb)
S1_COLS  = ["VV", "VH", "VH_VV_Ratio"]          # 3 radar features
S2_COLS  = ["NDVI", "NDWI", "NDRE", "EVI",
             "SAVI", "MSAVI", "NDMI", "GNDVI"]                           # 9 optical features
LOOKBACK = 120   # days
MAX_T    = 40   # max timesteps kept per sample

print(f"{'='*60}")
print(f"  ABLATION MODE : {ABLATION_MODE.upper()}")
print(f"  S1 features   : {S1_COLS}")
print(f"  S2 features   : {S2_COLS}")
print(f"  CUDA          : {torch.cuda.is_available()}")
print(f"{'='*60}")

  ABLATION MODE : S2
  S1 features   : ['VV', 'VH', 'VH_VV_Ratio']
  S2 features   : ['NDVI', 'NDWI', 'NDRE', 'EVI', 'SAVI', 'MSAVI', 'NDMI', 'GNDVI']
  CUDA          : True


## Cell 3 — Labels & Plot-Level Split

    Identical `GroupShuffleSplit(SEED=42, 70/15/15)` as model_3_presto.ipynb for fair comparison.

In [ ]:
gt = pd.read_csv(gt_path)
gt["date"] = pd.to_datetime(gt["date"])

stage_names  = sorted(gt["stage_name"].dropna().unique())
stage_to_idx = {n: i for i, n in enumerate(stage_names)}
idx_to_stage = {i: n for n, i in stage_to_idx.items()}
gt["stage_idx"] = gt["stage_name"].map(stage_to_idx)
num_classes = len(stage_names)
print(f"Classes ({num_classes}): {stage_to_idx}")

# Leakage-aware plot-level split (identical to model_3_presto.ipynb)
plot_df = pd.DataFrame({"plot_id": sorted(gt["plot_id"].unique())})

gss = GroupShuffleSplit(1, test_size=0.15, random_state=SEED)
tv_idx, te_idx = next(gss.split(plot_df, groups=plot_df["plot_id"]))
tv_df      = plot_df.iloc[tv_idx].reset_index(drop=True)
test_plots = set(plot_df.iloc[te_idx]["plot_id"])

gss2 = GroupShuffleSplit(1, test_size=0.1765, random_state=SEED)
tr_idx, va_idx = next(gss2.split(tv_df, groups=tv_df["plot_id"]))
train_plots = set(tv_df.iloc[tr_idx]["plot_id"])
val_plots   = set(tv_df.iloc[va_idx]["plot_id"])

assert train_plots.isdisjoint(val_plots) and train_plots.isdisjoint(test_plots)

train_rows = gt[gt["plot_id"].isin(train_plots)]
val_rows   = gt[gt["plot_id"].isin(val_plots)]
test_rows  = gt[gt["plot_id"].isin(test_plots)]
print(f"Plots  train={len(train_plots)} val={len(val_plots)} test={len(test_plots)}")
print(f"Rows   train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")

Classes (5): {'Bare': 0, 'Growth': 1, 'Ripening': 2, 'Seedling': 3, 'Tillering': 4}
Plots  train=142 val=31 test=31
Rows   train=16330 val=3565 test=3565


## Cell 4 — Dataset

Returns per-group sequences ready for PhenoCropPresto.
Ablation is implemented by **zeroing out** the irrelevant modality's features
and setting all its mask positions to `True` (missing), so the Transformer ignores those tokens.

In [ ]:
class PhenoCropDataset(Dataset):
    """Returns per-group sequences for PhenoCropPresto, with ablation-mode masking."""

    def __init__(self, gt_df, s1_dir, s2_dir):
        self.gt = gt_df.sort_values(["plot_id", "date"]).reset_index(drop=True)
        self.s1_cache, self.s2_cache = {}, {}
        for pid in sorted(self.gt["plot_id"].unique()):
            p1 = Path(s1_dir) / f"plot_{pid}_sar.csv"
            p2 = Path(s2_dir) / f"plot_{pid}_indices.csv"
            # Always load both — ablation zeroing happens in __getitem__
            if p1.exists():
                d = pd.read_csv(p1); d["date"] = pd.to_datetime(d["date"])
                self.s1_cache[pid] = d
            if p2.exists():
                d = pd.read_csv(p2); d["date"] = pd.to_datetime(d["date"])
                self.s2_cache[pid] = d

    def __len__(self): return len(self.gt)

    def _window(self, df, target_date, cols):
        """Extract lookback window → (feats [T,C], days_ago [T], mask_valid [T])."""
        feats    = np.zeros((MAX_T, len(cols)), dtype=np.float32)
        days_arr = np.zeros(MAX_T,              dtype=np.int64)
        valid    = np.zeros(MAX_T,              dtype=bool)
        if df is None or df.empty:
            return feats, days_arr, valid
        for c in cols:
            if c not in df.columns: df[c] = 0.0
        start = target_date - pd.Timedelta(days=LOOKBACK)
        win = df[(df["date"] > start) & (df["date"] <= target_date)].sort_values("date")
        if win.empty:
            return feats, days_arr, valid
        n = min(len(win), MAX_T)
        feats[-n:]    = win[cols].fillna(0.0).to_numpy(dtype=np.float32)[-n:]
        d_ago = (target_date - win["date"]).dt.days.to_numpy(dtype=np.int64)
        days_arr[-n:] = d_ago[-n:]
        valid[-n:]    = True
        return feats, days_arr, valid

    def __getitem__(self, idx):
        row   = self.gt.iloc[idx]
        pid   = int(row["plot_id"])
        tdate = row["date"]
        label = int(row["stage_idx"])

        s1f, s1d, s1v = self._window(self.s1_cache.get(pid), tdate, S1_COLS)
        s2f, s2d, s2v = self._window(self.s2_cache.get(pid), tdate, S2_COLS)

        # ── Ablation: zero out the unused modality AND mark all its tokens as missing
        if ABLATION_MODE == "s1":
            # S1 only: zero out S2 features and mask all S2 timesteps
            s2f = np.zeros_like(s2f)
            s2v = np.zeros_like(s2v)   # all invalid → True mask → ignored by Transformer
        elif ABLATION_MODE == "s2":
            # S2 only: zero out S1 features and mask all S1 timesteps
            s1f = np.zeros_like(s1f)
            s1v = np.zeros_like(s1v)   # all invalid → True mask → ignored by Transformer

        # Use S2 days_ago as canonical time axis (more frequent);
        # where S2 is missing, fall back to S1 days
        days_ago = np.where(s2v, s2d, s1d)

        return {
            "s1":      torch.tensor(s1f),              # [T, 3]
            "s2":      torch.tensor(s2f),              # [T, 9]
            "days":    torch.tensor(days_ago).long(),  # [T]
            "s1_mask": torch.tensor(~s1v),             # [T]  True = missing
            "s2_mask": torch.tensor(~s2v),             # [T]  True = missing
            "label":   torch.tensor(label).long(),
        }


train_ds = PhenoCropDataset(train_rows, s1_dir, s2_dir)
val_ds   = PhenoCropDataset(val_rows,   s1_dir, s2_dir)
test_ds  = PhenoCropDataset(test_rows,  s1_dir, s2_dir)
print(f"Datasets: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")

s = train_ds[0]
for k, v in s.items():
    if hasattr(v, 'shape'): print(f"  {k}: {tuple(v.shape)} {v.dtype}")

Datasets: 16330 / 3565 / 3565
  s1: (40, 3) torch.float32
  s2: (40, 8) torch.float32
  days: (40,) torch.int64
  s1_mask: (40,) torch.bool
  s2_mask: (40,) torch.bool
  label: () torch.int64


## Cell 5 — PhenoCrop-Presto Architecture (identical to model_3_presto.ipynb)

In [ ]:
# ── Positional encoding utilities (from Presto, unchanged) ──────────────────

def sinusoid_table(n_pos: int, d_hid: int) -> torch.Tensor:
    """Sinusoidal positional encoding [n_pos, d_hid]."""
    pos = torch.arange(n_pos).float().unsqueeze(1)          # [P, 1]
    dim = torch.arange(d_hid).float().unsqueeze(0)          # [1, D]
    angles = pos / torch.pow(10000, 2 * (dim // 2) / d_hid) # [P, D]
    angles[:, 0::2] = torch.sin(angles[:, 0::2])
    angles[:, 1::2] = torch.cos(angles[:, 1::2])
    return angles


def month_table(d_hid: int) -> torch.Tensor:
    """Sinusoidal month encoding [12, d_hid] (Jan=0 … Dec=11)."""
    assert d_hid % 2 == 0
    angles = torch.arange(12).float() / 12 * 2 * math.pi    # [12]
    sin_t  = torch.sin(angles).unsqueeze(1).expand(-1, d_hid // 2)
    cos_t  = torch.cos(angles).unsqueeze(1).expand(-1, d_hid // 2)
    return torch.cat([sin_t, cos_t], dim=-1)                 # [12, d_hid]


# ── Transformer Block (identical to Presto Block) ────────────────────────────

class TransformerBlock(nn.Module):
    def __init__(self, d: int, n_heads: int, mlp_ratio: float = 2.0, drop: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        self.mlp   = nn.Sequential(
            nn.Linear(d, int(d * mlp_ratio)), nn.GELU(), nn.Dropout(drop),
            nn.Linear(int(d * mlp_ratio), d), nn.Dropout(drop),
        )

    def forward(self, x, key_padding_mask=None):
        # x: [B, N, d]   key_padding_mask: [B, N]  True=ignore
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


# ── PhenoCropPresto Encoder ────────────────────────────────────────────────

class PhenoCropPresto(nn.Module):
    """
    Presto-style encoder adapted for pheno_crop_v2.

    Positional embedding = pos(days_ago) + channel_embed + month_embed
    Split into 3 components (same ratio as original Presto):
      50% sinusoidal days-ago  (replaces Presto's fixed positional embed)
      25% learnable channel id (S1 vs S2)
      25% sinusoidal month     (calendar seasonality)
    """

    BAND_GROUPS = {"S1": len(S1_COLS), "S2": len(S2_COLS)}  # {name: n_feats}
    N_GROUPS    = len(BAND_GROUPS)  # 2
    MAX_DAYS    = 120               # clamp days_ago to this

    def __init__(
        self,
        d_model:    int   = 128,
        depth:      int   = 4,
        n_heads:    int   = 8,
        mlp_ratio:  float = 2.0,
        drop:       float = 0.1,
        num_classes: int  = 5,
    ):
        super().__init__()
        assert d_model % 4 == 0, "d_model must be divisible by 4"
        self.d_model = d_model

        # ── Embedding dimension split (Presto ratios: 50% pos, 25% ch, 25% month)
        self.d_pos   = d_model // 2
        self.d_ch    = d_model // 4
        self.d_month = d_model // 4

        # ── Per-group input projections (Presto's eo_patch_embed)
        self.group_proj = nn.ModuleDict({
            name: nn.Linear(n_feats, d_model)
            for name, n_feats in self.BAND_GROUPS.items()
        })

        # ── Channel embedding: one scalar per group  (Presto's channel_embed)
        self.channel_embed = nn.Embedding(self.N_GROUPS, self.d_ch)

        # ── Fixed sinusoidal encodings (non-trainable, like Presto)
        pos_tab   = sinusoid_table(self.MAX_DAYS + 1, self.d_pos)   # [121, d_pos]
        month_tab = month_table(self.d_month)                        # [12,  d_month]
        self.register_buffer("pos_tab",   pos_tab)    # not saved as param
        self.register_buffer("month_tab", month_tab)

        # ── Transformer encoder blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, mlp_ratio, drop)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(d_model)

        # ── Classification head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(d_model // 2, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def _pos_embed(self, days_ago: torch.Tensor) -> torch.Tensor:
        """days_ago [B,T] → sinusoidal positional encoding [B,T,d_pos]"""
        idx = days_ago.clamp(0, self.MAX_DAYS)          # [B, T]
        return self.pos_tab[idx]                         # [B, T, d_pos]

    def _month_embed(self, days_ago: torch.Tensor, ref_date_month: int = 1) -> torch.Tensor:
        """
        Approximate the calendar month for each timestep using days_ago.
        month idx = (ref_month - days_ago_in_months) % 12
        returns [B, T, d_month]
        """
        months_ago = (days_ago // 30).clamp(0, 11)      # [B, T]
        month_idx  = (ref_date_month - months_ago) % 12 # [B, T]
        return self.month_tab[month_idx]                 # [B, T, d_month]

    def forward(
        self,
        s1:      torch.Tensor,   # [B, T,  3]
        s2:      torch.Tensor,   # [B, T,  9]
        days:    torch.Tensor,   # [B, T]   days-ago per timestep
        s1_mask: torch.Tensor,   # [B, T]   True = missing S1
        s2_mask: torch.Tensor,   # [B, T]   True = missing S2
    ) -> torch.Tensor:
        B, T, _ = s1.shape
        device   = s1.device

        # ── Positional components (shared across groups)
        pos_enc   = self._pos_embed(days)            # [B, T, d_pos]
        month_enc = self._month_embed(days)          # [B, T, d_month]

        tokens_list, mask_list = [], []

        for g_idx, (group_name, feat_tensor, group_mask) in enumerate([
            ("S1", s1, s1_mask),
            ("S2", s2, s2_mask),
        ]):
            # Project features → [B, T, d_model]
            tok = self.group_proj[group_name](feat_tensor)

            # Channel embedding: same ID for all timesteps of this group
            ch_idx = torch.tensor(g_idx, device=device)
            ch_enc = self.channel_embed(ch_idx)              # [d_ch]
            ch_enc = ch_enc.unsqueeze(0).unsqueeze(0).expand(B, T, -1)  # [B,T,d_ch]

            # Add positional info to token (Presto-style concatenated embedding)
            full_pos = torch.cat([pos_enc, ch_enc, month_enc], dim=-1)  # [B,T,d_model]
            tok = tok + full_pos

            tokens_list.append(tok)
            mask_list.append(group_mask)  # [B, T]

        # Concatenate groups along sequence dim → [B, 2T, d_model]
        x     = torch.cat(tokens_list, dim=1)
        # key_padding_mask [B, 2T]: True on positions to ignore
        kpm   = torch.cat(mask_list, dim=1)

        # Guard: if an entire row is masked, unmask last position per group
        # to avoid NaN in attention softmax
        all_masked = kpm.all(dim=1)  # [B]
        if all_masked.any():
            kpm = kpm.clone()
            kpm[all_masked, -1] = False

        # ── Transformer blocks
        for blk in self.blocks:
            x = blk(x, key_padding_mask=kpm)
        x = self.norm(x)  # [B, 2T, d_model]

        # ── Masked mean-pool (ignore padded positions)
        valid = (~kpm).unsqueeze(-1).float()              # [B, 2T, 1]
        pooled = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)  # [B, d_model]

        return self.classifier(pooled)                    # [B, num_classes]


# ── Quick sanity check ────────────────────────────────────────────────────
model = PhenoCropPresto(d_model=128, depth=4, n_heads=8, num_classes=num_classes)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")

# Forward pass test
dummy_s1  = torch.randn(4, MAX_T, len(S1_COLS))
dummy_s2  = torch.randn(4, MAX_T, len(S2_COLS))
dummy_d   = torch.randint(0, 90, (4, MAX_T))
dummy_m1  = torch.zeros(4, MAX_T, dtype=torch.bool)
dummy_m2  = torch.zeros(4, MAX_T, dtype=torch.bool)
out = model(dummy_s1, dummy_s2, dummy_d, dummy_m1, dummy_m2)
print(f"Output shape: {out.shape}  ✓")  # [4, 5]

Total params:     540,485
Trainable params: 540,485
Output shape: torch.Size([4, 5])  ✓


## Cell 6 — DataLoaders & Class Weights

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Batches: {len(train_loader)} / {len(val_loader)} / {len(test_loader)}")


def class_weights(ds, nc):
    labels = [ds[i]["label"].item() for i in range(len(ds))]
    counts = torch.bincount(torch.tensor(labels), minlength=nc)
    counts = counts.clamp(min=1).float()
    w = len(labels) / (nc * counts)
    return w.clamp(max=10.0)


cw = class_weights(train_ds, num_classes)
print(f"Class weights: {dict(enumerate(cw.numpy().round(3)))}")

Batches: 256 / 56 / 56
Class weights: {0: np.float32(1.958), 1: np.float32(0.907), 2: np.float32(1.316), 3: np.float32(0.961), 4: np.float32(0.63)}


## Cell 7 — Training & Evaluation Functions

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            s1      = batch["s1"].to(device)
            s2      = batch["s2"].to(device)
            days    = batch["days"].to(device)
            s1_mask = batch["s1_mask"].to(device)
            s2_mask = batch["s2_mask"].to(device)
            labels  = batch["label"].to(device)

            logits = model(s1, s2, days, s1_mask, s2_mask)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.detach().cpu().numpy())

    n       = max(len(all_true), 1)
    acc     = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return total_loss / n, acc, macro_f1, all_true, all_pred


def train(model, train_loader, val_loader, cw,
          epochs=100, lr=1e-3, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    model  = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=cw.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5
    )

    best_f1, wait, best_state = -1.0, 0, None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc, va_f1, _, _ = run_epoch(model, val_loader,   criterion, None,      device)
        scheduler.step(va_f1)

        if va_f1 > best_f1:
            best_f1, wait = va_f1, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss {tr_loss:.4f} Acc {tr_acc*100:.2f}% F1 {tr_f1:.4f} | "
            f"Val Loss {va_loss:.4f} Acc {va_acc*100:.2f}% F1 {va_f1:.4f}"
        )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}. Best val F1: {best_f1:.4f}")
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, device


def evaluate(model, loader, split, device):
    criterion = nn.CrossEntropyLoss()
    loss, acc, f1, y_true, y_pred = run_epoch(model, loader, criterion, None, device)
    labels = list(range(num_classes))
    names  = [idx_to_stage[i] for i in labels]

    print(f"\n[{split}] Loss: {loss:.4f} | Acc: {acc*100:.2f}% | Macro-F1: {f1:.4f}")
    recalls = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    for i, n, r in zip(labels, names, recalls):
        print(f"  [{i}] {n}: recall={r:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, target_names=names, zero_division=0))
    return loss, acc, f1

## Cell 8 — Train PhenoCropPresto

In [ ]:
model = PhenoCropPresto(
    d_model     = 128,
    depth       = 4,      # transformer layers
    n_heads     = 8,
    mlp_ratio   = 2.0,
    drop        = 0.1,
    num_classes = num_classes,
)

trained_model, device = train(
    model, train_loader, val_loader, cw,
    epochs=100, lr=1e-3, patience=15,
)

Device: cuda
Epoch 001 | Train Loss 0.8686 Acc 60.07% F1 0.5873 | Val Loss 0.7907 Acc 62.66% F1 0.6228
Epoch 002 | Train Loss 0.7370 Acc 65.79% F1 0.6450 | Val Loss 0.7721 Acc 63.03% F1 0.6198
Epoch 003 | Train Loss 0.6722 Acc 67.94% F1 0.6669 | Val Loss 0.7027 Acc 71.42% F1 0.7012
Epoch 004 | Train Loss 0.6501 Acc 68.78% F1 0.6755 | Val Loss 0.7517 Acc 71.64% F1 0.7094
Epoch 005 | Train Loss 0.6360 Acc 69.13% F1 0.6802 | Val Loss 0.7543 Acc 68.53% F1 0.6825
Epoch 006 | Train Loss 0.6205 Acc 70.09% F1 0.6885 | Val Loss 0.8046 Acc 68.84% F1 0.6893
Epoch 007 | Train Loss 0.6090 Acc 70.46% F1 0.6915 | Val Loss 0.7725 Acc 69.20% F1 0.6895
Epoch 008 | Train Loss 0.5920 Acc 71.44% F1 0.7007 | Val Loss 0.8049 Acc 68.92% F1 0.6842
Epoch 009 | Train Loss 0.5843 Acc 71.56% F1 0.7040 | Val Loss 0.7469 Acc 71.81% F1 0.7179
Epoch 010 | Train Loss 0.5697 Acc 72.77% F1 0.7149 | Val Loss 0.8289 Acc 69.85% F1 0.6924
Epoch 011 | Train Loss 0.5584 Acc 72.78% F1 0.7143 | Val Loss 0.8370 Acc 69.45% F1 0.68

## Cell 9 — Evaluate & Save Results

In [ ]:
val_loss,  val_acc,  val_f1  = evaluate(trained_model, val_loader,  "Validation",         device)
test_loss, test_acc, test_f1 = evaluate(trained_model, test_loader, "Test (Unseen Plots)", device)

print(f"\n{'='*55}")
print(f"  [{ABLATION_MODE.upper()}] Summary")
print(f"  Val  Macro-F1 : {val_f1:.4f}")
print(f"  Test Macro-F1 : {test_f1:.4f}")
print(f"{'='*55}")

# Save model weights
save_path = OUTPUT_DIR / f"ablation_presto_{ABLATION_MODE}.pth"
torch.save({
    "model_state_dict": trained_model.state_dict(),
    "ablation_mode":    ABLATION_MODE,
    "stage_to_idx":     stage_to_idx,
    "idx_to_stage":     idx_to_stage,
    "num_classes":      num_classes,
    "val_macro_f1":     val_f1,
    "test_macro_f1":    test_f1,
    "hparams":          {"d_model": 128, "depth": 4, "n_heads": 8},
}, save_path)
print(f"Model saved: {save_path}")

# Save results JSON
results = {
    "ablation_mode": ABLATION_MODE,
    "val_macro_f1":  round(float(val_f1),  4),
    "test_macro_f1": round(float(test_f1), 4),
}
out_path = OUTPUT_DIR / f"ablation_presto_{ABLATION_MODE}_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved: {out_path}")


[Validation] Loss: 1.0960 | Acc: 72.65% | Macro-F1: 0.7318
  [0] Bare: recall=0.8028
  [1] Growth: recall=0.6581
  [2] Ripening: recall=0.7726
  [3] Seedling: recall=0.7151
  [4] Tillering: recall=0.7249
              precision    recall  f1-score   support

        Bare       0.71      0.80      0.75       360
      Growth       0.73      0.66      0.69       781
    Ripening       0.63      0.77      0.69       774
    Seedling       0.89      0.72      0.79       723
   Tillering       0.73      0.72      0.73       927

    accuracy                           0.73      3565
   macro avg       0.74      0.73      0.73      3565
weighted avg       0.74      0.73      0.73      3565


[Test (Unseen Plots)] Loss: 0.5155 | Acc: 77.28% | Macro-F1: 0.7765
  [0] Bare: recall=0.9548
  [1] Growth: recall=0.7583
  [2] Ripening: recall=0.9532
  [3] Seedling: recall=0.6877
  [4] Tillering: recall=0.7282
              precision    recall  f1-score   support

        Bare       0.80      0.95    

## Final Ablation Table

Run this cell after completing **all 3 modes** (`s1s2`, `s1`, `s2`).
It loads the saved JSON files and prints the full comparison table.

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("./models")
modes      = ["s1s2", "s1", "s2"]
col_labels = ["S1+S2 (full)", "S1-only", "S2-only"]

all_results = {}
for m in modes:
    p = OUTPUT_DIR / f"ablation_presto_{m}_results.json"
    if p.exists():
        with open(p) as f:
            all_results[m] = json.load(f)
    else:
        print(f"[WARNING] {p} not found — run with ABLATION_MODE='{m}' first.")

if len(all_results) == 0:
    print("No results found yet. Run the full notebook for each ABLATION_MODE first.")
else:
    W = 18
    print(f"\n{'='*70}")
    print("  PhenoCrop-Presto ABLATION STUDY — Test Macro-F1  (delta vs S1+S2)")
    print(f"{'='*70}")
    header = f"  {'Metric':<28}" + "".join(f"{lbl:>{W}}" for lbl in col_labels)
    print(header)
    print("-" * 70)

    base_val  = all_results.get("s1s2", {}).get("val_macro_f1",  None)
    base_test = all_results.get("s1s2", {}).get("test_macro_f1", None)

    for metric_key, metric_label in [("val_macro_f1", "Val Macro-F1"),
                                     ("test_macro_f1", "Test Macro-F1")]:
        row = f"  {metric_label:<28}"
        base = all_results.get("s1s2", {}).get(metric_key)
        for i, m in enumerate(modes):
            val = all_results.get(m, {}).get(metric_key)
            if val is None:
                row += f"{'—':>{W}}"
            elif i == 0:
                row += f"{val:>{W}.4f}"
            else:
                delta = val - base if base else 0
                cell  = f"{val:.4f} ({delta:+.3f})"
                row  += f"{cell:>{W}}"
        print(row)

    print("=" * 70)
    missing = [m for m in modes if m not in all_results]
    if missing:
        print(f"  ⚠ Still needed: {missing} — re-run with ABLATION_MODE set accordingly.")
    else:
        print("  ✓ All 3 modes complete.")
    print("=" * 70)

[WARNING] models/ablation_presto_s1s2_results.json not found — run with ABLATION_MODE='s1s2' first.

  PhenoCrop-Presto ABLATION STUDY — Test Macro-F1  (delta vs S1+S2)
  Metric                            S1+S2 (full)           S1-only           S2-only
----------------------------------------------------------------------
  Val Macro-F1                                 —   0.6931 (+0.000)   0.7318 (+0.000)
  Test Macro-F1                                —   0.6602 (+0.000)   0.7765 (+0.000)
  ⚠ Still needed: ['s1s2'] — re-run with ABLATION_MODE set accordingly.
